# BDG Topology Analysis

Compares Behavioral Dependency Graph structure during attack vs benign phases.
Produces the centrality delta visualization supporting PCEPS feature f14.

In [ ]:
# Cell 1: Load BDG snapshots from evaluation runs
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import glob

RESULTS_DIR = Path('research/datasets/raw')
DATASET_DIR = Path('research/datasets/phantom-v1')

traces = pd.read_parquet(DATASET_DIR / 'traces.parquet')
labels = pd.read_parquet(DATASET_DIR / 'labels.parquet')

# Load BDG snapshot JSON files (exported by run_all_scenarios via PHANTOM API)
bdg_snapshots = []
for f in sorted(glob.glob(str(RESULTS_DIR / 'bdg_snapshots' / '*.json'))):
    try:
        with open(f) as fh:
            bdg_snapshots.append(json.load(fh))
    except Exception:
        pass

if not bdg_snapshots:
    print("No BDG snapshot files found in research/datasets/raw/bdg_snapshots/.")
    print("Run evaluation with --export-bdg-snapshots flag to generate them.")
    print("Using traces.parquet edge data for proxy graph metrics.")
    USE_TRACE_PROXY = True
else:
    print(f"Loaded {len(bdg_snapshots)} BDG snapshots.")
    USE_TRACE_PROXY = False


In [ ]:
# Cell 2: Graph metrics comparison — node count, edge count, density

if USE_TRACE_PROXY:
    # Use edge columns from traces as a proxy for BDG structure
    edge_traces = traces[
        traces['edge_src_purl'].notna() & traces['edge_dst_purl'].notna()
    ].copy()

    def graph_metrics(df):
        nodes = pd.unique(pd.concat([df['edge_src_purl'], df['edge_dst_purl']]))
        edges = df[['edge_src_purl', 'edge_dst_purl']].drop_duplicates()
        n, e = len(nodes), len(edges)
        density = (2 * e) / (n * (n - 1)) if n > 1 else 0
        return {'node_count': n, 'edge_count': e, 'density': density}

    metrics_by_phase = {}
    for phase in edge_traces['phase'].unique():
        sub = edge_traces[edge_traces['phase'] == phase]
        metrics_by_phase[phase] = graph_metrics(sub)

    gdf = pd.DataFrame(metrics_by_phase).T
    print("BDG Graph Metrics by Phase (trace-derived proxy):")
    print(gdf.round(4))
else:
    records = []
    for snap in bdg_snapshots:
        records.append({
            'snapshot_id': snap.get('snapshot_id'),
            'phase': snap.get('phase'),
            'attack_family': snap.get('attack_family'),
            'label': snap.get('label', 0),
            'node_count': snap.get('node_count', 0),
            'edge_count': snap.get('edge_count', 0),
            'density': snap.get('density', 0),
        })
    snap_df = pd.DataFrame(records)
    print(snap_df.groupby('phase')[['node_count', 'edge_count', 'density']].describe().round(4))


In [ ]:
# Cell 3: Centrality delta visualization

# edge_weight column from traces serves as a proxy for PageRank weight.
# In the real system, graph_centrality_delta (PCEPS f14) is computed by
# the causal engine and stored in phantom_attribution_confidence.

if USE_TRACE_PROXY:
    edge_attack = traces[(traces['label'] == 1) & (traces['edge_weight'].notna())]
    edge_benign = traces[(traces['label'] == 0) & (traces['edge_weight'].notna())]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].hist(edge_benign['edge_weight'].dropna(), bins=30, alpha=0.6,
                 color='#3498db', label='Benign', density=True)
    axes[0].hist(edge_attack['edge_weight'].dropna(), bins=30, alpha=0.6,
                 color='#e74c3c', label='Attack', density=True)
    axes[0].set_xlabel('BDG Edge Weight')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Edge Weight Distribution: Attack vs Benign')
    axes[0].legend()

    # Centrality proxy: group by purl and sum edge weights (in-degree proxy)
    purl_weight_attack = edge_attack.groupby('edge_dst_purl')['edge_weight'].sum().sort_values(ascending=False).head(15)
    purl_weight_benign = edge_benign.groupby('edge_dst_purl')['edge_weight'].sum().sort_values(ascending=False).head(15)

    delta_idx = purl_weight_attack.index.union(purl_weight_benign.index)
    delta = (
        purl_weight_attack.reindex(delta_idx, fill_value=0)
        - purl_weight_benign.reindex(delta_idx, fill_value=0)
    ).sort_values(ascending=False).head(10)

    delta.plot(kind='barh', ax=axes[1], color='#e74c3c')
    axes[1].axvline(0, color='black', linewidth=0.5)
    axes[1].set_xlabel('Centrality Delta (Attack - Benign)')
    axes[1].set_title('Top 10 PURLs by Centrality Increase\n(Attack vs Benign Phase)')
    axes[1].set_yticklabels([p[:40] for p in delta.index], fontsize=8)

    plt.tight_layout()
    plt.savefig('research/evaluation/results/bdg_centrality_delta.pdf', dpi=150, bbox_inches='tight')
    plt.show()
    print("Key finding: the attack target PURL should rank #1 in centrality delta.")


In [ ]:
# Cell 4: Degree distribution (power-law check)

if USE_TRACE_PROXY:
    all_edge_data = traces[traces['edge_src_purl'].notna()].copy()

    # Out-degree per PURL (how many unique destinations)
    out_deg = all_edge_data.groupby('edge_src_purl')['edge_dst_purl'].nunique()
    in_deg  = all_edge_data.groupby('edge_dst_purl')['edge_src_purl'].nunique()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, deg, label in [(axes[0], out_deg, 'Out-degree'), (axes[1], in_deg, 'In-degree')]:
        counts = deg.value_counts().sort_index()
        ax.loglog(counts.index, counts.values, 'o-', color='#2ecc71', markersize=4)
        ax.set_xlabel(label)
        ax.set_ylabel('Count (log scale)')
        ax.set_title(f'BDG {label} Distribution (log-log)')
        ax.grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.savefig('research/evaluation/results/bdg_degree_distribution.pdf', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Max out-degree: {out_deg.max()} | Max in-degree: {in_deg.max()}")
    print(f"Mean out-degree: {out_deg.mean():.2f} | Mean in-degree: {in_deg.mean():.2f}")


In [ ]:
# Cell 5: Path length from workload nodes to drift_event nodes

# In PHANTOM's BDG, drift_event nodes should be 1-2 hops from the
# workload node during an attack. During benign scenarios, no drift_event
# nodes should appear. This cell validates the BDG structural claim.

import networkx as nx

if USE_TRACE_PROXY:
    attack_edges = traces[
        (traces['label'] == 1) &
        traces['edge_src_purl'].notna() &
        traces['edge_dst_purl'].notna()
    ]
    G = nx.DiGraph()
    for _, row in attack_edges.iterrows():
        G.add_edge(row['edge_src_purl'], row['edge_dst_purl'],
                   weight=row['edge_weight'] or 1.0)

    print(f"Attack BDG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    print(f"Weakly connected components: {nx.number_weakly_connected_components(G)}")

    # Identify nodes that appear only in attack phase (drift candidates)
    benign_nodes = set(traces[traces['label'] == 0]['edge_src_purl'].dropna()) |                    set(traces[traces['label'] == 0]['edge_dst_purl'].dropna())
    attack_only = {n for n in G.nodes() if n not in benign_nodes and n}
    print(f"\nAttack-only nodes (potential drift_event nodes): {len(attack_only)}")
    for n in list(attack_only)[:10]:
        print(f"  {n}")
